# Recommandation d'hôtels — Multilingue hybride (Hugging Face + TF‑IDF + Numérique + Géo)

Ce notebook construit un moteur de recommandation d'hôtels multilingue et hybride utilisant toutes les colonnes du dataset (texte, numériques, géolocalisation et URL Google Maps) et renvoie une sortie en paragraphe avec lien cliquable.

## Modèles Hugging Face multilingues (liens)
- [intfloat/multilingual-e5-base](https://huggingface.co/intfloat/multilingual-e5-base) — excellent pour la recherche sémantique (préfixes 'query:'/'passage:')
- [sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2) — léger et rapide
- [sentence-transformers/distiluse-base-multilingual-cased-v2](https://huggingface.co/sentence-transformers/distiluse-base-multilingual-cased-v2)
- [sentence-transformers/LaBSE](https://huggingface.co/sentence-transformers/LaBSE)
- [Alibaba-NLP/gte-multilingual-base](https://huggingface.co/Alibaba-NLP/gte-multilingual-base)
- [BAAI/bge-m3](https://huggingface.co/BAAI/bge-m3)

Par défaut on charge un modèle rapide, modifiez la variable MODEL_NAME dans la section Configuration pour en tester un autre.

In [1]:
# Installation (exécuter si nécessaire)
import sys
!{sys.executable} -m pip install -U sentence-transformers torch scikit-learn scipy numpy pandas


  Using cached filelock-3.20.0-py3-none-any.whl.metadata (2.1 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2025.10.0-py3-none-any.whl.metadata (10 kB)
  Using cached setuptools-80.9.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached pyyaml-6.0.3-cp313-cp313-win_amd64.whl.metadata (2.4 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached markupsafe-3.0.3-cp313-cp313-win_amd64.whl.metadata (2.8 kB)
  Using cached anyio-4.11.0-py3-none-any.whl.metadata (4.1 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
   ---------------------------------------- 0.0/110.9 MB ? eta -:--:--
   - -------------------------------------- 5.2/110.9 MB 24.8 MB/s eta 0:00:05
 


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Imports et configuration
import os, math
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
from sentence_transformers import SentenceTransformer
from IPython.display import display, HTML

# Données train / test générées par votre pipeline
DATASET_PREFIX = 'hotels_maroc_30000_equilibres'
TRAIN_DATA_PATH = f'{DATASET_PREFIX}_clean_train.csv'
TEST_DATA_PATH  = f'{DATASET_PREFIX}_clean_test.csv'

# Choisir un modèle multilingue pré-entraîné
MODEL_NAME = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
EMB_TRAIN_PATH = f"embeddings_{DATASET_PREFIX}_{MODEL_NAME.split('/')[-1]}_train.npy"  # cache embeddings train
EMB_TEST_PATH  = f"embeddings_{DATASET_PREFIX}_{MODEL_NAME.split('/')[-1]}_test.npy"   # cache embeddings test
# Si vous changez de modèle, pensez à régénérer les caches

c:\AI_DATA\DATA1\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Chargement du dataset train/test
df_train = pd.read_csv(TRAIN_DATA_PATH, sep=';')
df_test  = pd.read_csv(TEST_DATA_PATH, sep=';')
print('Train:', df_train.shape, '| Test:', df_test.shape)
df_train.head(2)

Train: (24000, 18) | Test: (6000, 18)


,Ville,Type,Saison_ideale,Activites,Description,Budget_estime,Score_utilisateur,Duree_suggeree,Accessibilite,Securite_confort,Hotel,Latitude,Longitude,GoogleMap_URL,Services,Rating,Prix_nuit,full_text
0,taza,hôtel d'affaires,automne,"musique traditionnelle, monuments historiques,...","découvrez palace taza royal, un hôtel d'affair...",66DH/nuit,bien (3.8/5),2-3 nuits,centre-ville accessible,zone touristique,palace taza royal,34.217611,-3.996690,https://www.google.com/maps/search/Palace+Taza...,"piscine, navette aéroport, salle de sport, par...",3.8,66,taza | hôtel d'affaires | automne | musique tr...
1,larache,hôtel spa,printemps,"artisanat local, monuments historiques, dégust...","découvrez auberge palace, un hôtel spa d'excep...",360DH/nuit,bien (3.7/5),1 semaine,transports publics,sécurité privée,auberge palace,35.192770,-6.160019,https://www.google.com/maps/search/Auberge+Pal...,"piscine, réception 24h/24, navette aéroport, s...",3.7,360,larache | hôtel spa | printemps | artisanat lo...


In [4]:
# Construction du texte global à partir de toutes les colonnes textuelles (train/test)
text_cols = [
    'Ville','Hotel','Type','Saison_ideale','Activites','Description',
    'Accessibilite','Securite_confort','Services','Score_utilisateur','GoogleMap_URL'
]

def clean_text(s):
    if pd.isna(s):
        return ''
    s = str(s).replace('\n',' ').replace('\r',' ').strip().lower()
    return s

for c in text_cols:
    if c in df_train.columns:
        df_train[c] = df_train[c].apply(clean_text)
    if c in df_test.columns:
        df_test[c] = df_test[c].apply(clean_text)

cols_train = [c for c in text_cols if c in df_train.columns]
cols_test  = [c for c in text_cols if c in df_test.columns]

df_train['full_text'] = df_train[cols_train].apply(lambda r: ' | '.join([x for x in r if x]), axis=1)
df_test['full_text']  = df_test[cols_test].apply(lambda r: ' | '.join([x for x in r if x]), axis=1)

print('Exemple full_text (train):', df_train['full_text'].iloc[0][:300])
print('Exemple full_text (test) :', df_test['full_text'].iloc[0][:300])

Exemple full_text (train): taza | palace taza royal | hôtel d'affaires | automne | musique traditionnelle, monuments historiques, artisanat local, visite de la médina, jardins traditionnels | découvrez palace taza royal, un hôtel d'affaires d'exception à taza. cet établissement allie tradition marocaine et confort moderne pou
Exemple full_text (test) : berkane | riad berkane view | resort | toute l'année | jardins traditionnels, dégustation culinaire, artisanat local, visite de la médina, cours de cuisine | découvrez riad berkane view, un resort d'exception à berkane. cet établissement allie tradition marocaine et confort moderne pour un séjour in


In [5]:
# TF-IDF lexical (fit sur train, transform sur test)
tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1,2), stop_words=None)
X_tfidf_train = tfidf.fit_transform(df_train['full_text'])
X_tfidf_test  = tfidf.transform(df_test['full_text'])
X_tfidf_train.shape, X_tfidf_test.shape

((24000, 2008), (6000, 2008))

In [6]:
# Chargement du modèle d'embeddings multilingue
model = SentenceTransformer(MODEL_NAME)
model_name_lower = MODEL_NAME.lower()
use_prefix = any(k in model_name_lower for k in ['e5', 'gte', 'bge'])
model


c:\AI_DATA\DATA1\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hp\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling b

SentenceTransformer(
  (0): Transformer({'max_seq_length': 128, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
)

In [7]:
# Embeddings sur le corpus d'entraînement (train)
if os.path.exists(EMB_TRAIN_PATH):
    embeddings_train = np.load(EMB_TRAIN_PATH)
    print('Embeddings train chargés:', embeddings_train.shape)
else:
    texts_train = df_train['full_text'].tolist()
    corpus_texts = [('passage: ' + t) if use_prefix else t for t in texts_train]
    try:
        embeddings_train = model.encode(corpus_texts, batch_size=64, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True)
    except TypeError:
        embeddings_train = model.encode(corpus_texts, batch_size=64, convert_to_numpy=True)
        norms = np.linalg.norm(embeddings_train, axis=1, keepdims=True) + 1e-12
        embeddings_train = embeddings_train / norms
    np.save(EMB_TRAIN_PATH, embeddings_train)
embeddings_train.shape

Batches: 100%|██████████| 375/375 [28:11<00:00,  4.51s/it]


(24000, 384)

In [8]:
# Embeddings sur le corpus de test (test)
if os.path.exists(EMB_TEST_PATH):
    embeddings_test = np.load(EMB_TEST_PATH)
    print('Embeddings test chargés:', embeddings_test.shape)
else:
    texts_test = df_test['full_text'].tolist()
    corpus_texts_test = [('passage: ' + t) if use_prefix else t for t in texts_test]
    try:
        embeddings_test = model.encode(corpus_texts_test, batch_size=64, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True)
    except TypeError:
        embeddings_test = model.encode(corpus_texts_test, batch_size=64, convert_to_numpy=True)
        norms = np.linalg.norm(embeddings_test, axis=1, keepdims=True) + 1e-12
        embeddings_test = embeddings_test / norms
    np.save(EMB_TEST_PATH, embeddings_test)
embeddings_test.shape

Batches: 100%|██████████| 94/94 [06:33<00:00,  4.19s/it]


(6000, 384)

In [9]:
# Préparation des colonnes numériques et géographiques (corpus d'entraînement)
rating_train = pd.to_numeric(df_train.get('Rating'), errors='coerce').fillna(0.0).to_numpy()
rating_train_norm = np.clip(rating_train / 5.0, 0.0, 1.0)
price_train = pd.to_numeric(df_train.get('Prix_nuit'), errors='coerce').fillna(0.0).to_numpy()
lat_train = pd.to_numeric(df_train.get('Latitude'), errors='coerce').to_numpy()
lon_train = pd.to_numeric(df_train.get('Longitude'), errors='coerce').to_numpy()

p_low, p_high = np.nanpercentile(price_train, [5,95])
price_range = float(max(p_high - p_low, 1.0))

def price_closeness_train(target):
    if target is None:
        return np.zeros_like(price_train, dtype=float)
    d = np.abs(price_train - float(target))
    return 1.0 - np.minimum(d/price_range, 1.0)

def geo_score_train(q_lat, q_lon, cap_km=50.0):
    if (q_lat is None) or (q_lon is None):
        return np.zeros_like(rating_train_norm)
    R = 6371.0
    lat1 = np.radians(q_lat)
    lat2 = np.radians(lat_train)
    dlat = np.radians(lat_train - q_lat)
    dlon = np.radians(lon_train - q_lon)
    a = np.sin(dlat/2.0)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2.0)**2
    c = 2*np.arctan2(np.sqrt(a), np.sqrt(1-a))
    dist = R*c
    dist = np.where(np.isfinite(dist), dist, 1e9)
    s = 1.0 - np.minimum(dist/float(cap_km), 1.0)
    s = np.where(np.isfinite(s), s, 0.0)
    return s

def compute_lexical_scores_train(q):
    v = tfidf.transform([q])
    return linear_kernel(v, X_tfidf_train).ravel()

def encode_query(q):
    text = ('query: ' + q) if use_prefix else q
    try:
        vec = model.encode([text], convert_to_numpy=True, normalize_embeddings=True)
    except TypeError:
        vec = model.encode([text], convert_to_numpy=True)
        vec = vec / (np.linalg.norm(vec, axis=1, keepdims=True) + 1e-12)
    return vec[0]

In [12]:
# Recommandation hybride (dense + lexical + numeric + géo) — corpus d'entraînement uniquement

def recommend_hotels_train(query_text, top_k=10, city=None, lat_q=None, lon_q=None, price_target=None, min_rating=None, weights=None):
    if weights is None:
        weights = {'dense':0.55,'lex':0.30,'rating':0.10,'price':0.05,'geo':0.10}
    w = dict(weights)
    if price_target is None: w['price']=0.0
    if (lat_q is None) or (lon_q is None): w['geo']=0.0
    s = sum(w.values())
    if s<=0: w={'dense':1.0}; s=1.0
    for k in list(w.keys()): w[k] = w[k]/s

    q_vec = encode_query(query_text)
    dense_scores = embeddings_train.dot(q_vec)
    lex_scores = compute_lexical_scores_train(query_text)
    rating_scores = rating_train_norm.copy()
    price_scores = price_closeness_train(price_target)
    geo_scores = geo_score_train(lat_q, lon_q)

    final = (w.get('dense',0)*dense_scores + w.get('lex',0)*lex_scores +
             w.get('rating',0)*rating_scores + w.get('price',0)*price_scores +
             w.get('geo',0)*geo_scores)

    mask = np.ones(df_train.shape[0], dtype=bool)
    if city:
        mask &= df_train['Ville'].str.contains(str(city).strip().lower(), na=False)
    if min_rating is not None:
        mask &= (pd.to_numeric(df_train['Rating'], errors='coerce').fillna(0.0).to_numpy() >= float(min_rating))

    scores = final.copy()
    scores[~mask] = -1e9

    top_idx = np.argpartition(scores, -top_k)[-top_k:]
    top_idx = top_idx[np.argsort(scores[top_idx])[::-1]]
    res = df_train.iloc[top_idx].copy()
    res['__score__'] = scores[top_idx]
    return res


def as_paragraph(r):
    name = r.get('Hotel', '') or '(Hôtel)'
    city = r.get('Ville', '') or ''
    typ = r.get('Type', '') or ''
    rating_str = r.get('Rating', '')
    prix = r.get('Prix_nuit', '')
    desc = r.get('Description', '') or ''
    services = r.get('Services', '') or ''
    para = f"{name} à {city} est un {typ} noté {rating_str}/5, proposé à {prix} DH/nuit. {desc} Services: {services}."
    return para


def render_results(res_df, max_desc=350):
    blocks = []
    for _,r in res_df.iterrows():
        para = as_paragraph(r)
        if len(para)>max_desc: para = para[:max_desc].rstrip() + '...'
        url = r.get('GoogleMap_URL') or ''
        block = f"<div style='margin:10px 0'>\n<b>{r.get('Hotel','(Hôtel)')}</b> — {r.get('Ville','')} | Type: {r.get('Type','')} | Rating: {r.get('Rating','')}/5 | Prix: {r.get('Prix_nuit','')} DH/nuit<br>\n{para}<br>\n<a href='{url}' target='_blank'>Voir sur Google Maps</a>\n</div>"
        blocks.append(block)
    html = '<hr>'.join(blocks)
    display(HTML(html))

In [13]:
# Démo — requête en anglais (multilingue) — recherche dans le corpus d'entraînement
results_en = recommend_hotels_train(
    query_text="family friendly beachfront hotel with pool and spa",
    top_k=10,
    city=None,
    price_target=300,
    min_rating=4.0
)
render_results(results_en)

In [14]:
# Evaluation sur le jeu de test: requête = texte du test, corpus de recommandation = train

def recommend_for_test_item(test_row_idx, top_k=10, city=None, lat_q=None, lon_q=None, price_target=None, min_rating=None, weights=None):
    q = df_test['full_text'].iloc[int(test_row_idx)]
    return recommend_hotels_train(
        query_text=q,
        top_k=top_k,
        city=city,
        lat_q=lat_q,
        lon_q=lon_q,
        price_target=price_target,
        min_rating=min_rating,
        weights=weights,
    )

# Métrique simple: Hit@k par ville (la ville du test figure-t-elle dans les k premiers du train?)

def evaluate_test_by_city(N=200, k=10):
    N = min(int(N), len(df_test))
    hits = 0
    valid = 0
    for i in range(N):
        try:
            city = str(df_test['Ville'].iloc[i]).strip().lower()
            res = recommend_for_test_item(i, top_k=k)
            if len(res) == 0: 
                continue
            valid += 1
            if (res['Ville'].str.strip().str.lower() == city).any():
                hits += 1
        except Exception:
            pass
    return {
        'N_eval': valid,
        f'hit@{k}_same_city': (hits / valid) if valid else 0.0
    }


In [16]:
# Démo — requête en français (multilingue) — recherche dans le corpus d'entraînement
results_fr = recommend_hotels_train(
    query_text="hôtel familial près de la plage avec piscine et spa",
    top_k=10,
    city="agadir",
    price_target=300,
    min_rating=4.2
)
render_results(results_fr)